# 0. Data Exploration
Initial check of each source table before cleaning: size, data types, missing values, duplicates, outliers and category values. No data is modified in this notebook.

In [ ]:
import os
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)

# Folder containing the 7 CSV files
DATA_PATH = 'C:/Users/mishr/OneDrive/Documents/PYTHON_HACKATHON_Sep2026/Python_Hackathon_Sep_2026/cardiac_failure/'
KEY = 'inpatient_number'

csv_files = sorted(f for f in os.listdir(DATA_PATH) if f.endswith('.csv'))
print(f'{len(csv_files)} CSV files found:')
for f in csv_files:
    print('  ', f)

In [ ]:
def check_table(file_name, max_unique_to_show=15):
    df = pd.read_csv(DATA_PATH + file_name)
    print('=' * 80)
    print(f'TABLE: {file_name}')
    print('=' * 80)

    # 1. Size
    print(f'\n1. SHAPE: {df.shape[0]} rows x {df.shape[1]} columns')

    # 2. First rows
    print('\n2. FIRST 5 ROWS')
    display(df.head())

    # 3. Data types + nulls in one summary
    print('\n3. COLUMN SUMMARY (type, nulls, unique values)')
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'non_null': df.notnull().sum(),
        'null_count': df.isnull().sum(),
        'null_%': (df.isnull().mean() * 100).round(2),
        'unique': df.nunique()
    }).sort_values('null_%', ascending=False)
    display(summary)

    # 4. Duplicates
    print(f'\n4. FULL DUPLICATE ROWS: {df.duplicated().sum()}')
    if KEY in df.columns:
        print(f'   Unique {KEY}: {df[KEY].nunique()}  |  Duplicate IDs: {df[KEY].duplicated().sum()}  |  Null IDs: {df[KEY].isnull().sum()}')

    # 5. Problem flags
    print('\n5. FLAGS')
    high_null = summary[summary['null_%'] > 50].index.tolist()
    print(f'   Columns > 50% null: {high_null if high_null else "None"}')
    constant = [col for col in df.columns if df[col].nunique(dropna=True) <= 1]
    print(f'   Constant / single-value columns: {constant if constant else "None"}')
    # text columns that are mostly numbers (possible typing errors, e.g. "12.5 " or "<5")
    num_as_text = []
    for col in [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]:
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notnull().mean() > 0.8 * df[col].notnull().mean() and df[col].notnull().any():
            num_as_text.append(col)
    print(f'   Numbers stored as text: {num_as_text if num_as_text else "None"}')

    # 6. Numeric columns: stats + outliers (IQR rule)
    num_cols = [col for col in df.select_dtypes(include=np.number).columns if col != KEY]
    if num_cols:
        print('\n6. NUMERIC STATS')
        display(df[num_cols].describe().T)
        print('   Outlier count per column (outside 1.5 x IQR):')
        out = {}
        for col in num_cols:
            q1, q3 = df[col].quantile([0.25, 0.75])
            iqr = q3 - q1
            n = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
            if n > 0:
                out[col] = n
        print('  ', out if out else 'None')

    # 7. Categorical columns: value counts (look for typos / inconsistent spelling)
    cat_cols = [col for col in df.columns if col != KEY and (not pd.api.types.is_numeric_dtype(df[col]) or df[col].nunique() <= max_unique_to_show)]
    if cat_cols:
        print('\n7. VALUE COUNTS (categorical / low-unique columns)')
        for col in cat_cols:
            print(f'\n--- {col} ({df[col].nunique()} unique) ---')
            print(df[col].value_counts(dropna=False).head(max_unique_to_show).to_string())
    return df

## Labs

In [ ]:
labs = check_table('labs.csv')

## Demography

In [ ]:
demography = check_table('demography.csv')

## Hospitalization & Discharge

In [ ]:
hosp = check_table('hospitalization_discharge.csv')

## Cardiac Complications

In [ ]:
cardiac = check_table('cardiac_complications.csv')

## Responsiveness

In [ ]:
resp = check_table('responsivenes.csv')

## Patient History

In [ ]:
history = check_table('patienthistory.csv')

## Patient Prescriptions

In [ ]:
rx = check_table('patient_precriptions.csv')
# Extra check: this table has several rows per patient (one per drug)
print('\nDrugs per patient:')
print(rx.groupby(KEY)['drug_name'].count().describe())
print('\nSame drug listed twice for a patient:', rx.duplicated([KEY, 'drug_name']).sum())

## Cross-table ID check (run once, before merging)

In [ ]:
files = ['demography.csv', 'hospitalization_discharge.csv', 'labs.csv', 'cardiac_complications.csv',
         'patienthistory.csv', 'responsivenes.csv', 'patient_precriptions.csv']
ids = {f: set(pd.read_csv(DATA_PATH + f)[KEY]) for f in files}
all_ids = set.union(*ids.values())
print(f'Total unique patients across all tables: {len(all_ids)}\n')
for f, s in ids.items():
    missing = all_ids - s
    print(f'{f:35s} patients: {len(s):5d} | missing IDs: {sorted(missing)}')